In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive')        # the folder that CONTAINS soh_pipeline/
from soh_pipeline import Config, run_pipeline

cfg = Config(
    data_dir='/content/drive/MyDrive/cleaned_dataset/',   # has metadata.csv (and the data/ folder)
    output_dir='/content/drive/MyDrive/soh_processed',
)
result = run_pipeline(cfg)

d = result['datasets'][10]['train']       # window size 10
d['X'].shape, d['soh'].shape               # (n, 10, 128, 3), (n,)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[metadata] 2794 discharge cycles, 226 dropped (missing / zero / invalid capacity)
[metadata] dropping 16 batteries with first capacity < 1.5 Ah: B0033, B0034, B0036, B0038, B0039, B0040, B0041, B0045, B0049, B0050, B0051, B0052, B0053, B0054, B0055, B0056

            cycles  ambient_C  first_soh  last_soh  eol_idx  reaches_eol
battery_id                                                              
B0005          125         24       92.8      69.8    124.0         True
B0006          109         24      101.8      69.8    108.0         True
B0007          168         24       94.6      71.6      NaN        False
B0018           97         24       92.8      69.8     96.0         True
B0025           28         24       92.4      88.4      NaN        False
B0026           28         24       90.7      88.4      NaN        False
B0027           28         24 

Reading cycles:   0%|          | 0/978 [00:00<?, ?it/s]

[cycles] read 978 cycles, skipped 0
[split] train (12): B0006, B0007, B0018, B0025, B0027, B0028, B0029, B0032, B0043, B0044, B0046, B0048
[split] val (2): B0026, B0047
[split] test (4): B0005, B0030, B0031, B0042
[windows] N=20: {'train': 453, 'val': 9, 'test': 188}
[windows] N=10: {'train': 568, 'val': 20, 'test': 228}
[windows] N=5: {'train': 628, 'val': 30, 'test': 248}
[done] saved to /content/drive/MyDrive/soh_processed


((568, 10, 128, 3), (568,))

In [ ]:
import os
p = f'{out}/skipped_cycles.csv'
if os.path.getsize(p) > 1:
    print(pd.read_csv(p).reason.value_counts())
else:
    print('no cycles skipped')

print(pd.read_csv(f'{out}/battery_report.csv').to_string())
print(json.load(open(f'{out}/splits.json')))

for n in (20, 10, 5):
    z = np.load(f'{out}/windows_N{n}.npz', allow_pickle=True)
    for s in ('train', 'val', 'test'):
        X = z[f'{s}_X']
        print(n, s, X.shape, 'NaN:', np.isnan(X).any(), 'rul_valid:', int(z[f'{s}_rul_valid'].sum()))

no cycles skipped
   battery_id  cycles  ambient_C  first_soh  last_soh  eol_idx  reaches_eol
0       B0005     125         24       92.8      69.8    124.0         True
1       B0006     109         24      101.8      69.8    108.0         True
2       B0007     168         24       94.6      71.6      NaN        False
3       B0018      97         24       92.8      69.8     96.0         True
4       B0025      28         24       92.4      88.4      NaN        False
5       B0026      28         24       90.7      88.4      NaN        False
6       B0027      28         24       91.2      88.5      NaN        False
7       B0028      28         24       90.2      85.9      NaN        False
8       B0029      40         43       84.9      80.6      NaN        False
9       B0030      40         43       82.8      78.1      NaN        False
10      B0031      40         43       83.3      83.4      NaN        False
11      B0032      40         43       85.2      81.8      NaN        

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive')
from soh_pipeline import Config, load_saved, summarize_folds

cfg = Config(output_dir='/content/drive/MyDrive/soh_processed')
X, meta, fold_of = load_saved(cfg)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[folds] fold 0: B0044, B0025, B0029
[folds] fold 1: B0048, B0006, B0030, B0032
[folds] fold 2: B0043, B0005, B0027, B0031
[folds] fold 3: B0046, B0042, B0028
[folds] fold 4: B0047, B0018, B0026, B0007
 fold split  N  batteries  windows  rul_windows
    0   val 20          4      132           90
    0   val 10          4      169          107
    0   val  5          4      189          117
    0  test 20          3       61           31
    0  test 10          3       91           41
    0  test  5          3      106           46
    1   val 20          4      172          142
    1   val 10          4      212          162
    1   val  5          4      232          172
    1  test 20          4      132           90
    1  test 10          4      169          107
    1  test  5          4      189          117
    2   val 20          3       49           4